# UC2 — 01 Feature Engineering

**Project:** UC2 Inspector-Triggered Ticket-Purchase Fraud Detection

## What this notebook does

| Step | Description |
|---|---|
| 1 | Load the four input CSVs (activations, purchases, scans, enrichment) with tz-aware UTC timestamp validation |
| 2 | Derive activation gaps and emit one symbol per activation event (7-symbol vocabulary including `ACTIVATE_GATE`) |
| 3 | Compute Pattern HIGH (240 h, ≥ 3 gaps ≤ 15 s) and MEDIUM (168 h, ≥ 3 gaps ≤ 30 s) infraction counts |
| 4 | Join calendar enrichment so disruption-day exposure is captured per rider |
| 5 | Build the rider × feature matrix and prepare FIFO-capped symbol sequences for HMM training |

## Run order

1. This notebook → writes `feature_table.parquet` and `symbol_rows.parquet`
2. `02_UC2_HMM_Training.ipynb`
3. `03_UC2_Exercise3_Scoring.ipynb`
4. `04_UC2_Rule_Based_Validation.ipynb`


## 1 · Setup

In [1]:
import sys
from pathlib import Path

# expose the shared src/ modules
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd

from uc2_io       import UC2Paths, auto_paths, read_activations, read_purchases, read_scans, read_calendar, enrich_with_calendar
from uc2_symbols  import ActivationEvent, emit_symbol, SYMBOLS, SYMBOL_NAMES, N_SYMBOLS
from uc2_features import derive_pattern_counts, derive_timing_aggregates, prepare_sequences, MIN_EVENTS_FOR_HMM, MAX_SEQUENCE_LEN

pd.set_option('display.max_columns', 40)

# Data root resolves to pipeline/data/ (canonical) or a sibling data/
# folder if the CSVs have been placed outside the project root.
DATA_ROOT = None
for _base in (PROJECT_ROOT, PROJECT_ROOT.parent):
    _root = _base / 'data'
    if _root.exists():
        DATA_ROOT = _root
        break
if DATA_ROOT is None:
    raise FileNotFoundError(
        'No data/ folder found inside or next to pipeline/. '
        'Place the input CSVs in pipeline/data/ and re-run.'
    )

paths = auto_paths(local_base=DATA_ROOT)
paths.out_dir = PROJECT_ROOT / 'outputs'
paths.out_dir.mkdir(exist_ok=True)
print('Data root :', paths.base)
print('Output dir:', paths.out_dir)


Data root : /Users/saikrishnav/Desktop/Cap proj/pipeline/data
Output dir: /Users/saikrishnav/Desktop/Cap proj/pipeline/outputs


## 2 · Load input CSVs

All four CSVs are streamed through `uc2_io` which parses timestamps to tz-aware UTC and fails loudly if any row is un-parseable.


In [2]:
activations = read_activations(paths)
purchases   = read_purchases(paths)
scans       = read_scans(paths)

print('activations:', activations.shape)
print('purchases  :', purchases.shape)
print('scans      :', scans.shape,
      '- handheld:', (scans.scan_source == 'handheld').sum(),
      'gate:',      (scans.scan_source == 'gate').sum())

/Users/saikrishnav/Desktop/Cap proj/pipeline/src/uc2_io.py:239: UserWarning: _to_utc(activation_ts): 177 / 3,679,568 rows (0.005%) failed to parse — coerced to NaT
  df["activation_ts"] = _to_utc(df.pop("server_timestamp"), name="activation_ts")
/Users/saikrishnav/Desktop/Cap proj/pipeline/src/uc2_io.py:240: UserWarning: _to_utc(purchase_ts): 227 / 3,679,568 rows (0.006%) failed to parse — coerced to NaT
  df["purchase_ts"]   = _to_utc(df.pop("purchase_timestamp"), strict=False, name="purchase_ts")
/var/folders/vc/82p31fyx6vlfhq2xtb0t_08w0000gn/T/ipykernel_79385/2780473559.py:1: UserWarning: read_activations: dropped 177 rows with NaT activation_ts
  activations = read_activations(paths)
/Users/saikrishnav/Desktop/Cap proj/pipeline/src/uc2_io.py:265: UserWarning: _to_utc(purchase_ts): 74 / 1,749,838 rows (0.004%) failed to parse — coerced to NaT
  df["purchase_ts"] = _to_utc(df.pop("server_timestamp"), name="purchase_ts")
/var/folders/vc/82p31fyx6vlfhq2xtb0t_08w0000gn/T/ipykernel_79385

activations: (3679391, 5)
purchases  : (1749764, 3)
scans      : (6347748, 7) - handheld: 6347748 gate: 0


## 3 · Schema & timestamp checks

Every timestamp column is verified tz-aware UTC before downstream features are computed.


In [3]:
assert activations['activation_ts'].dt.tz is not None, 'activation_ts must be tz-aware'
assert purchases['purchase_ts'].dt.tz is not None
assert scans['scan_ts'].dt.tz is not None
assert set(scans['scan_source'].dropna().unique()) <= {'handheld', 'gate'}
print('timestamp + schema checks OK')

timestamp + schema checks OK


## 4 · Per-event timing features

For every activation we compute:

- `gap_s` — seconds since the rider's previous activation
- `seconds_to_handheld` — seconds to the next handheld scan on the same ticket (None if none)
- `seconds_to_gate` — seconds to the next gate scan on the same ticket (None if none)
- `seconds_since_purchase` — seconds since the most-recent purchase on the same ticket (None if none)

In [4]:
# sort once, per-account diff for gap_s
activations = activations.sort_values(['account_id', 'activation_ts']).reset_index(drop=True)
activations['gap_s'] = (
    activations.groupby('account_id', observed=True)['activation_ts']
               .diff().dt.total_seconds()
)

In [5]:
# Nearest-following validation scan per (account_id, ticket_id).
# merge_asof requires the 'on' key (activation_ts / scan_ts) globally sorted
# AND categorical by-keys with identical category sets — so we cast the
# by-keys to object only at the merge boundary. The big frames keep their
# categorical dtype (low RAM on the full 5.5 GB activations CSV).
handheld = scans[scans['scan_source'] == 'handheld'][['account_id', 'ticket_id', 'scan_ts']].copy()
gate     = scans[scans['scan_source'] == 'gate'    ][['account_id', 'ticket_id', 'scan_ts']].copy()

def nearest_follow(left: pd.DataFrame, right: pd.DataFrame, out_col: str) -> pd.DataFrame:
    left = left.sort_values('activation_ts', kind='mergesort').reset_index(drop=True)
    if right.empty:
        left[out_col] = pd.NA
        return left
    right = right.dropna(subset=['ticket_id']).sort_values('scan_ts', kind='mergesort').reset_index(drop=True)
    matchable   = left.dropna(subset=['ticket_id']).copy()
    unmatchable = left[left['ticket_id'].isna()].copy()
    for col in ('account_id', 'ticket_id'):
        if isinstance(matchable[col].dtype, pd.CategoricalDtype):
            matchable[col] = matchable[col].astype(object)
        if isinstance(right[col].dtype, pd.CategoricalDtype):
            right[col] = right[col].astype(object)
    merged = pd.merge_asof(
        matchable, right,
        by=['account_id', 'ticket_id'],
        left_on='activation_ts', right_on='scan_ts',
        direction='forward',
    )
    merged[out_col] = (merged['scan_ts'] - merged['activation_ts']).dt.total_seconds()
    merged = merged.drop(columns=['scan_ts'])
    unmatchable[out_col] = pd.NA
    combined = pd.concat([merged, unmatchable], ignore_index=True)
    return combined.sort_values(['account_id', 'activation_ts'], kind='mergesort').reset_index(drop=True)

activations = nearest_follow(activations, handheld, 'seconds_to_handheld')
activations = nearest_follow(activations, gate,     'seconds_to_gate')
print('seconds_to_handheld non-null:', activations['seconds_to_handheld'].notna().sum())
print('seconds_to_gate     non-null:', activations['seconds_to_gate'].notna().sum(),
      '(expected 0 in current dataset — no gate scans)')


/var/folders/vc/82p31fyx6vlfhq2xtb0t_08w0000gn/T/ipykernel_79385/4023178222.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([merged, unmatchable], ignore_index=True)


seconds_to_handheld non-null: 2942319
seconds_to_gate     non-null: 0 (expected 0 in current dataset — no gate scans)


In [6]:
# purchase_ts is already denormalised into the activations table — compute directly.
activations['seconds_since_purchase'] = (
    activations['activation_ts'] - activations['purchase_ts']
).dt.total_seconds()
print('seconds_since_purchase non-null:', activations['seconds_since_purchase'].notna().sum())
activations[['account_id','activation_ts','ticket_id','purchase_ts','seconds_since_purchase','seconds_to_handheld','seconds_to_gate']].head()

seconds_since_purchase non-null: 3679187


,account_id,activation_ts,ticket_id,purchase_ts,seconds_since_purchase,seconds_to_handheld,seconds_to_gate
0,f92c1997d4b42580110f1d7f80548531d4824ec7ed7aa4...,2025-07-01 08:00:28.883000+00:00,805b25900ea932308eedaceaf7cdc727a70e9b3ee3579a...,2025-07-01 08:00:22+00:00,6.883,45284.577,<NA>
1,eb8b6d28ba29492f5ed2fe88c5b2f3db0cf053573a332f...,2025-07-01 08:00:59.182000+00:00,a783f3a55ac64d05ec3680d0c7454fe6c24d96a0042c10...,2025-07-01 05:58:47+00:00,7332.182,NaN,<NA>
2,e3eb064e91249389f4c9d50aa91069891e42ae2dd849f3...,2025-07-01 08:01:07.475000+00:00,f48aa29ac6898b4c7532aa39830de951bb3acb35130539...,2025-07-01 08:00:57+00:00,10.475,159.564,<NA>
3,ab72fc59cef3d98d17513342eb205cf61ea8ed676a6b87...,2025-07-01 08:02:34.114000+00:00,eb114bda579d43c6230a2b07d39327a087d6fdd176858e...,2025-06-26 23:25:35+00:00,376619.114,3.822,<NA>
4,596840cf5605dd03ff9635399cbef45610447e28f6cb29...,2025-07-01 08:03:32.523000+00:00,f6cfe9757ee1b365050005c8e6a2c8d5966f3fe5a830f7...,2025-07-01 08:01:59+00:00,93.523,5.035,<NA>


## 5 · Emit HMM symbols

Uses the 7-symbol vocabulary from `uc2_symbols.emit_symbol`. The vocabulary includes `ACTIVATE_GATE` for the gate-validation pathway; repeat-offender signal is represented at the rider-aggregate level rather than as an observation symbol.


In [7]:
# Vectorised symbol emission — mirrors uc2_symbols.emit_symbol rules 1-6.
# ~100x faster than .apply(row_to_event, axis=1) on wide data.
def _f(col): return pd.to_numeric(activations[col], errors='coerce').to_numpy(dtype=float)
sh = _f('seconds_to_handheld')
sg = _f('seconds_to_gate')
sp = _f('seconds_since_purchase')

symbol_id = np.full(len(activations), SYMBOLS['OTHER_HANDHELD_PATTERN'], dtype=int)
m1 = (~np.isnan(sp)) & (sp <= 60) & (~np.isnan(sh)) & (sh <= 15)                     ; symbol_id[m1] = SYMBOLS['PURCHASE_THEN_ACTIVATE_FAST']
m2 = (~m1) & (~np.isnan(sh)) & (sh >= 16) & (sh <= 30)                                ; symbol_id[m2] = SYMBOLS['ACTIVATE_GAMING_THRESHOLD']
m3 = (~m1) & (~m2) & (~np.isnan(sh)) & (sh <= 15)                                     ; symbol_id[m3] = SYMBOLS['ACTIVATE_FAST_HANDHELD']
m4 = (~m1) & (~m2) & (~m3) & (~np.isnan(sg)) & (sg <= 120)                            ; symbol_id[m4] = SYMBOLS['ACTIVATE_GATE']
m5 = (~m1) & (~m2) & (~m3) & (~m4) & (~np.isnan(sh)) & (sh <= 300)                    ; symbol_id[m5] = SYMBOLS['ACTIVATE_SLOW_HANDHELD']
m6 = (~m1) & (~m2) & (~m3) & (~m4) & (~m5) & np.isnan(sh) & np.isnan(sg)              ; symbol_id[m6] = SYMBOLS['NO_HANDHELD_FOLLOWUP']

activations['symbol_id']   = symbol_id
activations['symbol_name'] = activations['symbol_id'].map(SYMBOL_NAMES)
activations['symbol_name'].value_counts()

symbol_name
ACTIVATE_FAST_HANDHELD         918162
OTHER_HANDHELD_PATTERN         805332
ACTIVATE_SLOW_HANDHELD         798833
NO_HANDHELD_FOLLOWUP           737072
ACTIVATE_GAMING_THRESHOLD      303108
PURCHASE_THEN_ACTIVATE_FAST    116884
Name: count, dtype: int64

## 6 · Aggregate rider-level features

In [8]:
# Pattern infraction counts
gap_df = activations[['account_id', 'activation_ts', 'gap_s']].copy()
gap_df['activate_ts'] = gap_df['activation_ts'].astype('int64') // 10**9
pattern_counts = derive_pattern_counts(gap_df[['account_id', 'activate_ts', 'gap_s']])

# Timing aggregates (mean/median/min/max, near_threshold_ratio)
timing_agg = derive_timing_aggregates(gap_df[['account_id', 'gap_s']])

# Volume + symbol counts
volume = (
    activations.groupby('account_id', observed=True)
               .agg(n_activations=('activation_ts', 'size'),
                    first_activation=('activation_ts', 'min'),
                    last_activation =('activation_ts', 'max'))
)
volume['tenure_days'] = (volume['last_activation'] - volume['first_activation']).dt.total_seconds() / 86400

symbol_wide = (
    activations.groupby(['account_id', 'symbol_name'], observed=True).size()
               .unstack(fill_value=0)
               .add_prefix('n_')
)

feature_table = (
    volume.join(pattern_counts, how='left')
          .join(timing_agg,     how='left')
          .join(symbol_wide,    how='left')
          .fillna(0)
)
feature_table['infraction_rate']      = (feature_table['max_infractions_240h'] + feature_table['max_infractions_168h']) / feature_table['n_activations'].clip(lower=1)
feature_table['repeat_offender_flag'] = feature_table['max_infractions_240h'] >= 3
feature_table['hmm_eligible']         = feature_table['n_activations'] >= MIN_EVENTS_FOR_HMM
feature_table.head()

/Users/saikrishnav/Desktop/Cap proj/pipeline/src/uc2_features.py:200: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return events.groupby("account_id", observed=True).apply(_per_group)


,n_activations,first_activation,last_activation,tenure_days,max_infractions_240h,max_infractions_168h,has_pattern_high,has_pattern_medium,mean_gap_seconds,median_gap_seconds,min_gap_seconds,max_gap_seconds,near_threshold_ratio,n_ACTIVATE_FAST_HANDHELD,n_ACTIVATE_GAMING_THRESHOLD,n_ACTIVATE_SLOW_HANDHELD,n_NO_HANDHELD_FOLLOWUP,n_OTHER_HANDHELD_PATTERN,n_PURCHASE_THEN_ACTIVATE_FAST,infraction_rate,repeat_offender_flag,hmm_eligible
account_id,,,,,,,,,,,,,,,,,,,,,,
000001ac9ce714e6ca33c85a863a170b1623c37b3ce6aea4001f202fd1c4dc6d,3,2025-09-18 17:13:47.815000+00:00,2025-10-19 20:05:03.602000+00:00,31.118933,0,0,False,False,1.344338e+06,1.344338e+06,32151.597,2656524.190,0.0,1,0,0,0,2,0,0.0,False,False
00003a39dcfb23483e21c46ab7460ff917be23f3092a76cb26920bf7519b4d54,1,2025-07-22 01:13:30.498000+00:00,2025-07-22 01:13:30.498000+00:00,0.000000,0,0,False,False,0.000000e+00,0.000000e+00,0.000,0.000,0.0,0,0,0,1,0,0,0.0,False,False
00006f9d9ee12b3ca036399c3e804fc361a26241a356205fbb2f816333b47260,3,2025-07-11 21:25:40.909000+00:00,2025-08-16 02:46:09.018000+00:00,35.222548,0,0,False,False,1.521614e+06,1.521614e+06,11791.593,3031436.516,0.0,2,0,0,0,1,0,0.0,False,False
0000729888b6b8504cd3c205e20345755f27668be6a571fb338aa27fab71100b,2,2025-09-13 19:37:04.364000+00:00,2025-09-13 23:36:06.538000+00:00,0.165997,0,0,False,False,1.434217e+04,1.434217e+04,14342.174,14342.174,0.0,0,0,0,0,2,0,0.0,False,False
0000e47494724306b76ab68a1bc72eb0ff928aa3800c71b5c3bf8ac0c6fcee92,3,2025-12-26 21:53:32.639000+00:00,2025-12-26 21:53:41.653000+00:00,0.000104,0,0,False,False,4.507000e+00,4.507000e+00,4.288,4.726,0.0,0,0,0,0,3,0,0.0,False,False


## 7 · Enrichment joins

Joins the calendar-of-events reference table onto each rider as mean exposure to schedule impacts, maintenance, events, school days, and holidays. This protects the shortlist from false positives driven by legitimate gap-looking behaviour on disruption days.


In [9]:
try:
    calendar = read_calendar(paths)
    activations_cal = enrich_with_calendar(activations, calendar)
    calendar_flags = [c for c in activations_cal.columns if c.startswith('has_')]
    if calendar_flags:
        calendar_agg = activations_cal.groupby('account_id', observed=True)[calendar_flags].mean()
        feature_table = feature_table.join(calendar_agg.add_prefix('cal_'), how='left').fillna(0)
        print('calendar joined:', calendar_flags)
    else:
        print('calendar_of_events.csv has no has_* columns — skipping enrichment')
except FileNotFoundError:
    print('calendar_of_events.csv not in data dir — skipping enrichment join')

calendar joined: ['has_schedule_impact', 'has_maintenance_impact', 'has_event_impact', 'has_school_impact', 'has_holiday_impact']


## 8 · Sequence preparation

FIFO cap of 30 symbols per rider keeps long-tenured accounts from dominating the HMM likelihood.


In [10]:
symbol_rows = activations[['account_id', 'activation_ts', 'symbol_id']].copy()
symbol_rows['activate_ts'] = symbol_rows['activation_ts'].astype('int64') // 10**9
symbol_rows = symbol_rows[['account_id', 'activate_ts', 'symbol_id']]

batch = prepare_sequences(symbol_rows, min_events=MIN_EVENTS_FOR_HMM, max_len=MAX_SEQUENCE_LEN)
print(f'eligible riders: {len(batch.account_ids):,}')
print(f'total symbols:   {batch.concatenated.size:,}')
print(f'seq length pct (p50/p90/p99): {np.percentile(batch.lengths, [50, 90, 99])}')

eligible riders: 98,241
total symbols:   1,767,777
seq length pct (p50/p90/p99): [16. 30. 30.]


## 9 · Save artifacts

In [11]:
def _save_df(df, stem):
    """Parquet if pyarrow available; pickle as universal fallback."""
    pq = paths.out_dir / f'{stem}.parquet'
    pkl = paths.out_dir / f'{stem}.pkl'
    try:
        df.to_parquet(pq)
        print('  parquet:', pq)
    except Exception as exc:
        print(f'  parquet skipped ({exc.__class__.__name__}) — using pickle')
    df.to_pickle(pkl)
    print('  pickle :', pkl)

print('feature_table:')
_save_df(feature_table, 'feature_table')
print('symbol_rows:')
_save_df(symbol_rows,  'symbol_rows')

np.savez(paths.out_dir / 'sequences.npz',
         account_ids=np.asarray(batch.account_ids),
         concatenated=batch.concatenated,
         lengths=batch.lengths)
print('sequences.npz:', paths.out_dir / 'sequences.npz')

feature_table:
  parquet: /Users/saikrishnav/Desktop/Cap proj/pipeline/outputs/feature_table.parquet
  pickle : /Users/saikrishnav/Desktop/Cap proj/pipeline/outputs/feature_table.pkl
symbol_rows:
  parquet: /Users/saikrishnav/Desktop/Cap proj/pipeline/outputs/symbol_rows.parquet
  pickle : /Users/saikrishnav/Desktop/Cap proj/pipeline/outputs/symbol_rows.pkl
sequences.npz: /Users/saikrishnav/Desktop/Cap proj/pipeline/outputs/sequences.npz
